In [ ]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from ydata_profiling import ProfileReport
from sklearn.feature_selection import chi2

In [ ]:
#Loading the data
student = pd.read_csv("Student_Academic_Data.csv", sep = ';')

In [ ]:
#Removing the spaces in the heading with _
student.columns = ['''Marital_status''','''Application_mode''','''Application_order''','''Course''','''"Daytime/evening_attendance"''','''Previous_qualification''','''Previous_qualification_(grade)''','''Nationality''','''Mothers_qualification''','''Fathers_qualification''','''Mothers_occupation''','''Fathers_occupation''','''Admission_grade''','''Displaced''','''Educational_special_needs''','''Debtor''','''Tuition_fees_up_to_date''','''Gender''','''Scholarship_holder''','''Age_at_enrollment''','''International''','''Curricular_units_1st_sem_(credited)''','''Curricular_units_1st_sem_(enrolled)''','''Curricular_units_1st_sem_(evaluations)''','''Curricular_units_1st_sem_(approved)''','''Curricular_units_1st_sem_(grade)''','''Curricular_units_1st_sem_(without_evaluations)''','''Curricular_units_2nd_sem_(credited)''','''Curricular_units_2nd_sem_(enrolled)''','''Curricular_units_2nd_sem_(evaluations)''','''Curricular_units_2nd_sem_(approved)''','''Curricular_units_2nd_sem_(grade)''','''Curricular_units_2nd_sem_(without_evaluations)''','''Unemployment_rate''','''Inflation_rate''','''GDP''','''Target''']

In [ ]:
student

In [ ]:
#Creating a Report on all the attributes
profile=ProfileReport(student, title="Profiling Report")

In [ ]:
profile

In [ ]:
#Checking for missing data
student.isnull().sum()

In [ ]:
#Checking the data types of each attribute
student.dtypes

In [ ]:
#Count the total number of our target
student['Target'].value_counts()

In [ ]:
#Removed all rows where students were currentlly enrolled as we are only looking for students that have graduated or dropped out
student = student[student.Target != 'Enrolled']
student['Target'].value_counts()

In [ ]:
#Convert our target attributes into numbers. 1=Graduate 2=Dropout
le = LabelEncoder()
student['Target'] = le.fit_transform(student['Target'])
student['Target'].value_counts()

In [ ]:
#Create a Data Frame with the variables that we want to focus on for prediction models later
student2 = student[['''"Daytime/evening_attendance"''', "Age_at_enrollment", "Mothers_qualification", "Fathers_qualification", "Target"]]

In [ ]:
#Remove GDP and Inflation_rate attibutes as they contains negative values
student = student.drop(["GDP","Inflation_rate"], axis=1)

In [ ]:
#Use Chi Square test for feature selection
x = student.drop('Target', axis = 1)
y = student.Target


f_score = chi2(x, y)
p_value = pd.Series(f_score[1],index = x.columns)
p_value = p_value.sort_values(ascending = True)
print(p_value)

In [ ]:
#Find the variables that are below a significance level of 0.05
significance_level = 0.05
for i in p_value.index:
    if p_value[i] > significance_level:
        print(i,':   There is not relationshp between the Independent variable and the Target Value')

In [ ]:
# Removing the variables that are not significant
student = student.drop(['''"Daytime/evening_attendance"''',"Fathers_qualification","Educational_special_needs","International", "Mothers_occupation", "Unemployment_rate" ], axis=1)

In [ ]:
#Splitting the data into traning (80%) and testing (20%)
x = student.drop('Target', axis = 1)
y = student.Target


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 7)

In [ ]:
#Balancing the data set using Synthetic Minority Oversampling Technique (SMOTE)
smote = SMOTE(random_state = 7)

x_train, y_train = smote.fit_resample(x_train, y_train.ravel())

In [ ]:
#building a decision tree model
dt = tree.DecisionTreeClassifier(random_state=7)
student_dt = dt.fit(x_train, y_train)

In [ ]:
#Make predictions with the decision tree
y_pred_dt = dt.predict(x_test)

In [ ]:
# Check the Matthews Correlation Coefficient (MCC) for the Decision tree Model
print("The MCC for the Decision Tree is: ", (matthews_corrcoef(y_test, y_pred_dt)))

In [ ]:
# check the accuracy of the decision tree
print(classification_report(y_test, y_pred_dt))
cf=confusion_matrix(y_test, y_pred_dt)
print ("Confusion Matrix")
print(cf)
tn, fp, fn, tp=cf.ravel()
print ("TP: ", tp,", FP: ", fp,", TN: ", tn,", FN:", fn)

In [ ]:
#Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize = (10, 5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title("Confusion Matrix for Decision Tree Model")
plt.show()

In [ ]:
#Creating a Naive Bayes Model
nb = MultinomialNB()

nb.fit(x_train, y_train)

y_pred_nb = nb.predict(x_test)

In [ ]:
# Check the Matthews Correlation Coefficient (MCC) for the Naive Bayes Model
print('The MCC for the Naive Bayes Model is: ', (matthews_corrcoef(y_test, y_pred_nb)))

In [ ]:
# check the accuracy of the Naive Bayes Model
print(classification_report(y_test, y_pred_nb))
cf=confusion_matrix(y_test, y_pred_nb)
print ("Confusion Matrix")
print(cf)
tn, fp, fn, tp=cf.ravel()
print ("TP: ", tp,", FP: ", fp,", TN: ", tn,", FN:", fn)

In [ ]:
#Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize = (10, 5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title("Confusion Matrix for Naive Bayes Model")
plt.show()

Below we will be testing prediciton models with only "Daytime/evening_attendance", "Age_at_enrollment", "Mothers_qualification", and "Fathers_qualification" as those are the variable that my research is focused on. we can then compare the results to the above 2 models that use most features.

In [ ]:
#Splitting the data into traning (80%) and testing (20%)
x = student2.drop('Target', axis = 1)
y = student2.Target


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 7)

In [ ]:
#Balancing the data set using Synthetic Minority Oversampling Technique (SMOTE)

x_train, y_train = smote.fit_resample(x_train, y_train.ravel())

In [ ]:
#building a Second decision tree model
dt = tree.DecisionTreeClassifier(random_state=7)
student2_dt = dt.fit(x_train, y_train)

In [ ]:
#Make predictions with the decision tree
y_pred_dt = dt.predict(x_test)

In [ ]:
# Check the Matthews Correlation Coefficient (MCC) for the Second Decision tree Model
print("The MCC for the Second Decision Tree is: ", (matthews_corrcoef(y_test, y_pred_dt)))

In [ ]:
# check the accuracy of the Second decision tree
print(classification_report(y_test, y_pred_dt))
cf=confusion_matrix(y_test, y_pred_dt)
print ("Confusion Matrix")
print(cf)
tn, fp, fn, tp=cf.ravel()
print ("TP: ", tp,", FP: ", fp,", TN: ", tn,", FN:", fn)

In [ ]:
#Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize = (10, 5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title("Confusion Matrix for Second Decision tree Model")
plt.show()

In [ ]:
#Creating a Second Naive Bayes Model
nb = MultinomialNB()

nb.fit(x_train, y_train)

y_pred_nb = nb.predict(x_test)

In [ ]:
# Check the Matthews Correlation Coefficient (MCC) for the Second Naive Bayes Model
print('The MCC for the Second Naive Bayes Model is: ', (matthews_corrcoef(y_test, y_pred_nb)))

In [ ]:
# check the accuracy of the Second Naive Bayes Model
print(classification_report(y_test, y_pred_nb))
cf=confusion_matrix(y_test, y_pred_nb)
print ("Confusion Matrix")
print(cf)
tn, fp, fn, tp=cf.ravel()
print ("TP: ", tp,", FP: ", fp,", TN: ", tn,", FN:", fn)

In [ ]:
#Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize = (10, 5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title("Confusion Matrix for Second Naive Bayes Model")
plt.show()